# **CODE-4:**


In [0]:
'''
CODE-4: CREATING A COMBINED DATA MATRIX FOR CPCB AND SATELLITE DATA
==========================================================================================

OBJECTIVE: MATCHING SATELLITE DATA AND CPCB DATA ACCORDING TO TIME 
           AND NUMBER OF DATA POINTS FOR A PARTICULAR PLACE
TO DO/RELOOK:
-> Length of delhi_sat_data_df, datamatrix_ground and datamatrix_comb should be same
-> Check for the column numbers in delhi_ground_data_df for place, pm2.5, RH and WS 
-> In datamatrix_comb both columns that show 'place' should match
-> Reset the index to concatenate sat data and ground data

INPUT  : pixel_count_sat_data, delhi_ground_df, delhi_sat_data_imputed
OUTPUT : datamatrix_comb

==========================================================================================
'''

In [0]:
import pandas as pd
import numpy as np

In [0]:
chennai_sat_data_imputed = pd.read_csv('/content/chennai_final_sat_data_imputed7.csv')
chennai_sat_data_imputed.head()

In [0]:
chennai_ground_df = pd.read_csv('/content/chennai_final_ground_data7.csv')
chennai_ground_df.columns

Index(['Unnamed: 0', 'year', 'month', 'day', 'hour', 'minute', 'From Date',
       'place', 'PM2.5', 'Temp', 'RH', 'PM10'],
      dtype='object')

In [0]:
# Dividing into year-wise data for satellite dataset
def year_wise(str, year):
     return chennai_sat_data_imputed[chennai_sat_data_imputed[str] == year]  
yr_2016s = year_wise('Year', 2016)
yr_2017s = year_wise('Year', 2017)
yr_2018s = year_wise('Year', 2018)
yr_2019s = year_wise('Year', 2019)
yr_2020s = year_wise('Year', 2020)

In [0]:
def pixel_count(df):
  pixel_year_count = [] 
  pixel_year_count = df.groupby(['Year','Month', 'Day','Hour','RoundedMinute','Place']).size().reset_index(name='count')
  return pixel_year_count

pixc_2016 = pixel_count(yr_2016s)
print("2016 - ",len(pixc_2016))
pixc_2017 = pixel_count(yr_2017s)
print("2017 - ",len(pixc_2017))
pixc_2018 = pixel_count(yr_2018s)
print("2018 - ",len(pixc_2018))
pixc_2019 = pixel_count(yr_2019s)
print("2019 - ", len(pixc_2019))
pixc_2020 = pixel_count(yr_2020s)
print("2020 - ", len(pixc_2020))


2016 -  164
2017 -  138
2018 -  142
2019 -  133
2020 -  147


In [0]:
#for cross checking
ample = chennai_ground_df.groupby(['place']).size().reset_index(name='count')
ample2 = chennai_sat_data_imputed.groupby(['Place']).size().reset_index(name='count')
ample.to_csv('ample.csv')
ample2.to_csv('ample2.csv')

In [0]:
#cross verifying place names
n=0
for i in range(len(ample['place'])):
  if ample['place'][i] == ample2['Place'][i]:
    n = n+1
  else:
    print(ample['place'][i])
print(n)

3


In [0]:
# Dividing into year-wise data for ground dataset
def year_wise(str, year):
     return chennai_ground_df[chennai_ground_df[str] == year]  
yr_2016g = year_wise('year', 2016)
yr_2017g = year_wise('year', 2017)
yr_2018g = year_wise('year', 2018)
yr_2019g = year_wise('year', 2019)
yr_2020g = year_wise('year', 2020)

In [0]:
print("2016 - ",len(yr_2016g))
print("2017 - ",len(yr_2017g))
print("2018 - ",len(yr_2018g))
print("2019 - ",len(yr_2019g))
print("2020 - ",len(yr_2020g))

2016 -  4368
2017 -  4320
2018 -  4320
2019 -  4320
2020 -  4368


In [0]:
def sat_similar_ground_matrix(pixel_count_sat_data, chennai_ground_df):

  chennai_ground_df  = chennai_ground_df.sort_values(['place','year','month','day','hour','minute'], ascending=[True, True, True, True, True, True])
  #changing the index column
  chennai_ground_df.reset_index(drop=True, inplace=True) 
  


  datamatrix_ground = pd.DataFrame()
  datamatrix_comb = pd.DataFrame()
  dmx_temp=pd.DataFrame()
  p=0 
  nsamp = 0 
  #to get the number(count) of chennai_new_df1 values from chennai_ground_data_df.iterrows for a particular time      
  for i_pc,row_pc in pixel_count_sat_data.iterrows():
      for i_cg, row_cg in chennai_ground_df.iterrows():
          #if all variables in these two dataframes are same then based on count of sat data, ground data will be entered into sat data  
          if (row_pc['Place'] == row_cg['place'] and row_pc['Year']==row_cg['year'] and row_pc['Month']==row_cg['month'] and row_pc['Day']==row_cg['day'] 
              and row_pc['Hour'] == row_cg['hour'] and row_pc['RoundedMinute']==row_cg['minute']): 
              dmx_temp = chennai_ground_df.iloc[i_cg:i_cg+int(row_pc["count"]),[8,9,10,7,6]] #all columns
              if(p==0):
                  datamatrix_ground = dmx_temp
              else:
                  datamatrix_ground = pd.concat([datamatrix_ground,dmx_temp])
              p=p+1
              nsamp = nsamp + 1
              print(nsamp)
              break
  print(p) #to know the number of iterations  
  return datamatrix_ground

Caution: Will take time to run

yr2016_comb will run for corresponding length of pixc_2016.
Similarly rest of the years

In [0]:
yr2016_ssgm = sat_similar_ground_matrix(pixc_2016, yr_2016g)
yr2017_ssgm = sat_similar_ground_matrix(pixc_2017, yr_2017g)
yr2018_ssgm = sat_similar_ground_matrix(pixc_2018, yr_2018g)
yr2019_ssgm = sat_similar_ground_matrix(pixc_2019, yr_2019g)
yr2020_ssgm = sat_similar_ground_matrix(pixc_2020, yr_2020g)

In [0]:
len(yr_2018s)

198

In [0]:
def combine_sat_ground(chennai_sat_data_imputed, datamatrix_ground):  
  datamatrix_ground.reset_index(drop=True, inplace=True) 
  chennai_sat_data_imputed.reset_index(drop=True, inplace=True) 
  datamatrix_comb = pd.concat([datamatrix_ground,chennai_sat_data_imputed],axis=1)
  return datamatrix_comb

In [0]:
yr2016_comb = combine_sat_ground(yr_2016s, yr2016_ssgm)
yr2017_comb = combine_sat_ground(yr_2017s, yr2017_ssgm)
yr2018_comb = combine_sat_ground(yr_2018s, yr2018_ssgm)
yr2019_comb = combine_sat_ground(yr_2019s, yr2019_ssgm)
yr2020_comb = combine_sat_ground(yr_2020s, yr2020_ssgm)

In [0]:
yr2020_comb

In [0]:
# chennai_sat_data_df
combined_year_dflist = [yr2016_comb, yr2017_comb,yr2018_comb ,yr2019_comb, yr2020_comb ]
combined = pd.DataFrame()
for df in combined_year_dflist:
    combined = combined.append(df)

In [0]:
combined.to_csv('chennai_final_combined_data7.csv')